# Đánh giá TextRank và ViT5

Luồng tám cell code: Drive → GitHub → thư viện → hồ sơ/dữ liệu → validation → chốt quyết định → test → tải kết quả.

## 1. Kết nối Drive và chọn run

In [1]:
from pathlib import Path
from google.colab import drive

evaluation_run_id = ""
validation_comparison_id = ""

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")
if not drive_root.is_dir():
    raise FileNotFoundError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")


Mounted at /content/drive


## 2. Lấy mã đánh giá từ GitHub

In [2]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = ""

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

remote = subprocess.run(
    ["git", "remote", "get-url", "origin"], cwd=repo_dir,
    capture_output=True, text=True, check=True,
)
if remote.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=repo_dir,
    capture_output=True, text=True, check=True,
)
if status.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
if target_commit == "":
    result = subprocess.run(
        ["git", "rev-parse", "origin/main"], cwd=repo_dir,
        capture_output=True, text=True, check=True,
    )
    target_commit = result.stdout.strip()
subprocess.run(["git", "checkout", "--detach", target_commit], cwd=repo_dir, check=True)
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
print("Commit đánh giá:", target_commit)


Commit đánh giá: 3759a9915e2ff71f1a6fd321ef875308cb6e4147


## 3. Cài thư viện

In [3]:
%pip uninstall -y -q diffusers torchvision timm fastai
%pip install -q "jedi==0.20.1"
%pip install -q -r /content/vietnamese-summarization/requirements.txt
%pip check


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
No broken requirements found.


## 4. Đọc hồ sơ, cấu hình và dữ liệu validation

Kiểm tra best đúng summary/config, manifest hiện tại.

In [4]:
import csv
import gc
import json
import shutil
from datetime import datetime, timezone
import torch
import src.evaluate
from app.demo_data import select_completed_run
from src.inference import load_summarizer, summarize_text
from src.metrics import score_summaries
from src.textRank import summarize_textrank
from src.tokenization import load_tokenization_source
from src.utils import load_train_config, load_generation_config, set_seed

run_profile, run_summary, checkpoint_dir = select_completed_run(drive_root, evaluation_run_id)
evaluation_run_id = run_profile["run_id"]
output_dir = drive_root / "outputs" / evaluation_run_id
runtime_dir = Path("/content/vietnamese_summarization_evaluation") / evaluation_run_id
runtime_dir.mkdir(parents=True, exist_ok=True)
config_path = runtime_dir / "train.json"
generation_path = runtime_dir / "generation.json"
config_path.write_text(json.dumps(run_profile["train_config"], ensure_ascii=False, indent=2), encoding="utf-8")
generation_path.write_text(json.dumps(run_profile["generation_config"], ensure_ascii=False, indent=2), encoding="utf-8")
train_config = load_train_config(config_path)
generation_config = load_generation_config(generation_path)

manifests = {}
for name in ["subset_manifest", "processed_manifest"]:
    current_path = drive_root / train_config["paths"][name]
    saved_path = output_dir / current_path.name
    if current_path.read_bytes() != saved_path.read_bytes():
        raise RuntimeError("Lỗi: Manifest hiện tại khác hồ sơ run: " + name)
    manifests[name] = json.loads(saved_path.read_text(encoding="utf-8"))

loaded_config, splits, source_files, subset_fingerprint = load_tokenization_source(drive_root, config_path)
if subset_fingerprint != run_profile["data"]["subset_manifest_sha256"]:
    raise RuntimeError("Lỗi: Tập con không khớp run đã chọn.")
validation_rows = splits["validation"]
print("Run đánh giá:", evaluation_run_id)
print("Best epoch:", run_summary["best_epoch"])
print("Best validation loss:", run_summary["best_validation_loss"])
print("Số mẫu validation:", len(validation_rows))
print("Giới hạn nguồn/đích:", train_config["max_source_length"], train_config["max_target_length"])
print("Cấu hình sinh ban đầu:", generation_config)


Run đánh giá: vit5_finetuning_20261113_071230_169031
Best epoch: 2
Best validation loss: 2.1058625596091947
Số mẫu validation: 201
Giới hạn nguồn/đích: 1024 128
Cấu hình sinh ban đầu: {'num_beams': 4, 'max_new_tokens': 128, 'do_sample': False}


## 5. Validation: greedy, beam và TextRank

In [5]:
from transformers import AutoTokenizer
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: Cần GPU để so sánh decoding ViT5.")
gc.collect()
torch.cuda.empty_cache()
comparison_report = src.evaluate.run_decoding_comparison(
    project_root=repo_dir,
    drive_root=drive_root,
    checkpoint_run_id=evaluation_run_id,
    config_path=config_path,
    generation_config_path=generation_path,
)
gc.collect()
torch.cuda.empty_cache()
comparison_dir = Path(comparison_report["output_dir"])
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir / "best", use_fast=False, local_files_only=True)
greedy_records = []
for line in (comparison_dir / "greedy_predictions.jsonl").read_text(encoding="utf-8").splitlines():
    greedy_records.append(json.loads(line))
if len(greedy_records) != len(validation_rows):
    raise RuntimeError("Lỗi: Không đủ dự đoán validation ViT5.")
records = []
references = []
predictions = []
textrank_config = None
for index, row in enumerate(validation_rows):
    result = summarize_textrank(row["document"], tokenizer, config_path)
    source_hash = src.evaluate.source_ids_fingerprint(result["used_input_ids"])
    if greedy_records[index]["id"] != row["id"] or greedy_records[index]["source_ids_sha256"] != source_hash:
        raise RuntimeError("Lỗi: Nguồn TextRank không khớp lượt validation ViT5.")
    if textrank_config is None:
        textrank_config = result["textrank_config"]
    elif textrank_config != result["textrank_config"]:
        raise RuntimeError("Lỗi: Cấu hình TextRank thay đổi giữa các mẫu.")
    records.append({
        "id": row["id"], "source": row["document"], "reference": row["summary"],
        "prediction": result["summary"], "source_ids_sha256": source_hash,
        "was_truncated": result["was_truncated"],
        "summary_was_truncated": result["summary_was_truncated"],
        "converged": result["converged"], "elapsed_seconds": result["elapsed_seconds"],
    })
    references.append(row["summary"])
    predictions.append(result["summary"])
scores = score_summaries(references, predictions)
with (comparison_dir / "textrank_predictions.jsonl").open("w", encoding="utf-8") as file:
    for index, record in enumerate(records):
        record["scores"] = scores["per_sample"][index]
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
textrank_report = {
    "checkpoint_run_id": evaluation_run_id,
    "comparison_id": comparison_report["comparison_id"],
    "train_config": train_config,
    "textrank_config": textrank_config,
    "validation_sample_ids": comparison_report["validation_sample_ids"],
    "sample_count": scores["sample_count"],
    "mean_f1": scores["mean_f1"], "mean_f1_percent": scores["mean_f1_percent"],
    "metric_settings": scores["settings"],
    "source": src.evaluate.record_evaluation_source(repo_dir, ["src/textRank.py", "notebooks/vietnamese_summarization_evaluation.ipynb"]),
    "same_source_ids_check_passed": True,
    "test_content_read": False,
}
(comparison_dir / "textrank_validation.json").write_text(json.dumps(textrank_report, ensure_ascii=False, indent=2), encoding="utf-8")
run_validation_dir = output_dir / "validation" / comparison_report["comparison_id"]
run_validation_dir.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(comparison_dir, run_validation_dir)
for method in ["greedy", "beam"]:
    print("ViT5", method, comparison_report["candidates"][method]["mean_f1_percent"])
print("TextRank:", textrank_report["mean_f1_percent"])
print("Đề xuất decoding:", comparison_report["proposed_method"])
print("Comparison ID:", comparison_report["comparison_id"])
validation_archive = shutil.make_archive(str(run_validation_dir), "zip", root_dir=run_validation_dir)
files.download(validation_archive)


Chạy khởi động, không tính vào thời gian: greedy
Chạy khởi động, không tính vào thời gian: beam
Bắt đầu: greedy Số mẫu: 201
greedy Đã xử lý: 25 / 201
greedy Đã xử lý: 50 / 201
greedy Đã xử lý: 75 / 201
greedy Đã xử lý: 100 / 201
greedy Đã xử lý: 125 / 201
greedy Đã xử lý: 150 / 201
greedy Đã xử lý: 175 / 201
greedy Đã xử lý: 200 / 201
greedy Đã xử lý: 201 / 201
greedy ROUGE F1 x 100: {'rouge1': 23.44445825196356, 'rouge2': 10.608910634461424, 'rougeL': 20.770173679490682}
Bắt đầu: beam Số mẫu: 201
beam Đã xử lý: 25 / 201
beam Đã xử lý: 50 / 201
beam Đã xử lý: 75 / 201
beam Đã xử lý: 100 / 201
beam Đã xử lý: 125 / 201
beam Đã xử lý: 150 / 201
beam Đã xử lý: 175 / 201
beam Đã xử lý: 200 / 201
beam Đã xử lý: 201 / 201
beam ROUGE F1 x 100: {'rouge1': 22.786412079368006, 'rouge2': 10.65147126659274, 'rougeL': 20.41185591571407}
Đã lưu báo cáo: /content/drive/.shortcut-targets-by-id/1pYCC4s6c4UIpyf6pvB0tIsvIlJTL48zs/vietnamese-summarization/outputs/decoding_comparison_20261113_090658_697046/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Chốt phương án trước khi mở test

In [ ]:
evaluate_test = False
selected_method = ""

if evaluate_test:
    if selected_method not in ["greedy", "beam"]:
        raise RuntimeError("Lỗi: Chọn greedy hoặc beam sau khi xem validation.")
    candidates = []
    if validation_comparison_id != "":
        if Path(validation_comparison_id).name != validation_comparison_id or validation_comparison_id in [".", ".."]:
            raise RuntimeError("Lỗi: Comparison ID không được chứa đường dẫn.")
        candidates.append(output_dir / "validation" / validation_comparison_id)
    else:
        candidates = sorted((output_dir / "validation").glob("decoding_comparison_*"), reverse=True)
    selected_comparison_dir = None
    for directory in candidates:
        if (directory / "decoding_comparison.json").is_file() and (directory / "textrank_validation.json").is_file():
            selected_comparison_dir = directory
            break
    if selected_comparison_dir is None:
        raise RuntimeError("Lỗi: Chưa có validation đầy đủ TextRank/ViT5. Hãy chạy cell 5.")
    comparison_report = json.loads((selected_comparison_dir / "decoding_comparison.json").read_text(encoding="utf-8"))
    textrank_report = json.loads((selected_comparison_dir / "textrank_validation.json").read_text(encoding="utf-8"))
    for report in [comparison_report, textrank_report]:
        if report["checkpoint_run_id"] != evaluation_run_id or report["train_config"] != train_config:
            raise RuntimeError("Lỗi: Validation không khớp run/cấu hình.")
    if textrank_report["comparison_id"] != comparison_report["comparison_id"]:
        raise RuntimeError("Lỗi: Hai báo cáo không thuộc cùng lượt validation.")
    expected_ids = []
    for row in validation_rows:
        expected_ids.append(row["id"])
    for report in [comparison_report, textrank_report]:
        if report["validation_sample_ids"] != expected_ids:
            raise RuntimeError("Lỗi: Validation không khớp ID dữ liệu.")
    for key in ["best_epoch", "best_global_step", "best_validation_loss"]:
        if comparison_report[key] != run_summary[key]:
            raise RuntimeError("Lỗi: Best trong validation không khớp summary.")
    current_source = src.evaluate.record_evaluation_source(repo_dir, ["src/textRank.py", "notebooks/vietnamese_summarization_evaluation.ipynb"])
    for filename in ["src/inference.py", "src/preprocessing.py", "src/metrics.py", "src/utils.py"]:
        recorded_hash = comparison_report["source"]["runtime_source_sha256"][filename]
        if current_source["runtime_source_sha256"][filename] != recorded_hash:
            raise RuntimeError("Lỗi: Mã suy luận/metric đã đổi sau validation. Hãy chạy lại cell 5.")
    selected_generation = comparison_report["candidates"][selected_method]["generation_config"]
    expected_textrank_config = textrank_report["textrank_config"]
    if current_source["runtime_source_sha256"]["src/textRank.py"] != textrank_report["source"]["runtime_source_sha256"]["src/textRank.py"]:
        raise RuntimeError("Lỗi: Mã TextRank đã đổi sau validation. Hãy đo lại validation.")
    created_at = datetime.now(timezone.utc)
    test_dir = output_dir / ("test_evaluation_" + created_at.strftime("%Y%m%d_%H%M%S_%f"))
    test_dir.mkdir(parents=True, exist_ok=False)
    decision = {
        "run_id": evaluation_run_id, "comparison_id": comparison_report["comparison_id"],
        "selected_method": selected_method, "generation_config": selected_generation,
        "textrank_config": expected_textrank_config,
        "selection_basis": "user selection after validation; before reading test",
        "created_at_utc": created_at.isoformat(),
    }
    selected_generation_path = test_dir / "generation.json"
    selected_generation_path.write_text(json.dumps(selected_generation, ensure_ascii=False, indent=2), encoding="utf-8")
    (test_dir / "decoding_selection.json").write_text(json.dumps(decision, ensure_ascii=False, indent=2), encoding="utf-8")
    for name in ["decoding_comparison.json", "textrank_validation.json"]:
        shutil.copyfile(selected_comparison_dir / name, test_dir / name)
    print("Run:", evaluation_run_id)
    print("Validation:", comparison_report["comparison_id"])
    print("Đã lưu quyết định:", selected_method)
else:
    print("Test chưa bật. Xem validation rồi chốt selected_method trước khi chạy test.")


## 7. Chấm TextRank và ViT5 trên test

In [ ]:
if evaluate_test:
    if not torch.cuda.is_available():
        raise RuntimeError("Lỗi: Cần GPU để đánh giá ViT5.")
    saved_decision = json.loads((test_dir / "decoding_selection.json").read_text(encoding="utf-8"))
    if saved_decision != decision or decision["run_id"] != evaluation_run_id or decision["selected_method"] != selected_method:
        raise RuntimeError("Lỗi: Quyết định đã thay đổi. Hãy chốt lại ở cell 6.")
    if load_generation_config(selected_generation_path) != decision["generation_config"]:
        raise RuntimeError("Lỗi: Cấu hình sinh đã đổi sau khi chốt.")
    if load_train_config(config_path) != train_config:
        raise RuntimeError("Lỗi: Cấu hình train tạm đã đổi. Hãy chạy lại phần thiết lập.")
    actual_source = src.evaluate.record_evaluation_source(repo_dir, ["src/textRank.py", "notebooks/vietnamese_summarization_evaluation.ipynb"])
    if actual_source != current_source:
        raise RuntimeError("Lỗi: Source đã đổi sau khi chốt. Hãy kiểm tra lại validation.")
    model, tokenizer, selection = load_summarizer(checkpoint_dir, config_path)
    for key, summary_key in [("epoch", "best_epoch"), ("global_step", "best_global_step"), ("validation_loss", "best_validation_loss")]:
        if selection[key] != run_summary[summary_key]:
            raise RuntimeError("Lỗi: Model best đã đổi sau khi chọn run.")
    set_seed(train_config["seed"])
    rows, test_metadata = src.evaluate.load_fixed_test_source(drive_root, {
        "train_config": train_config,
        "subset": manifests["subset_manifest"],
        "processed": manifests["processed_manifest"],
    })
    references = []
    textrank_predictions = []
    vit5_predictions = []
    records = []
    for index, row in enumerate(rows):
        textrank = summarize_textrank(row["document"], tokenizer, config_path)
        if textrank["textrank_config"] != decision["textrank_config"]:
            raise RuntimeError("Lỗi: Tham số TextRank khác validation đã chốt.")
        vit5 = summarize_text(row["document"], model, tokenizer, config_path, selected_generation_path)
        if textrank["used_input_ids"] != vit5["used_input_ids"]:
            raise RuntimeError("Lỗi: TextRank và ViT5 không dùng cùng nguồn.")
        if vit5["generation_config"] != decision["generation_config"]:
            raise RuntimeError("Lỗi: Cấu hình sinh thực tế khác quyết định.")
        references.append(row["summary"])
        textrank_predictions.append(textrank["summary"])
        vit5_predictions.append(vit5["summary"])
        records.append({
            "id": row["id"], "source": row["document"], "reference": row["summary"],
            "textrank": textrank["summary"], "vit5": vit5["summary"],
            "source_ids_sha256": src.evaluate.source_ids_fingerprint(textrank["used_input_ids"]),
            "input_token_count": vit5["input_token_count"],
            "used_input_token_count": vit5["used_input_token_count"],
            "source_was_truncated": vit5["was_truncated"],
            "textrank_summary_was_truncated": textrank["summary_was_truncated"],
            "textrank_converged": textrank["converged"],
            "textrank_seconds": textrank["elapsed_seconds"], "vit5_seconds": vit5["elapsed_seconds"],
        })
        if (index + 1) % 25 == 0 or index + 1 == len(rows):
            print("Đã xử lý:", index + 1, "/", len(rows))
    results = {}
    for method, predictions in [("textrank", textrank_predictions), ("vit5", vit5_predictions)]:
        scores = score_summaries(references, predictions)
        results[method] = {
            "sample_count": scores["sample_count"],
            "mean_f1": scores["mean_f1"], "mean_f1_percent": scores["mean_f1_percent"],
        }
        for index in range(len(records)):
            records[index][method + "_scores"] = scores["per_sample"][index]
    metrics = {
        "run_id": evaluation_run_id, "run_status": "completed",
        "checkpoint": str(checkpoint_dir / "best"),
        "best_epoch": selection["epoch"], "best_global_step": selection["global_step"],
        "best_validation_loss": selection["validation_loss"],
        "train_config": train_config, "decision": decision,
        "generation_config": decision["generation_config"], "textrank_config": decision["textrank_config"],
        "source": current_source, "test_metadata": test_metadata,
        "metric_settings": scores["settings"], "results": results,
        "test_previously_inspected_in_project": True,
        "test_used_for_tuning_this_evaluation": False,
        "token_budget_note": "TextRank content tokens; ViT5 new tokens may include EOS",
    }
    with (test_dir / "predictions.jsonl").open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")
    (test_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    with (test_dir / "comparison.csv").open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["method", "sample_count", "rouge1_f1_percent", "rouge2_f1_percent", "rougeL_f1_percent"])
        for method in ["textrank", "vit5"]:
            row = [method, results[method]["sample_count"]]
            for metric in ["rouge1", "rouge2", "rougeL"]:
                row.append(results[method]["mean_f1_percent"][metric])
            writer.writerow(row)
    for name in ["config.json", "training_summary.json"]:
        shutil.copyfile(output_dir / name, test_dir / name)
    for name in ["subset_manifest", "processed_manifest"]:
        filename = Path(train_config["paths"][name]).name
        shutil.copyfile(output_dir / filename, test_dir / filename)
    saved_metrics = json.loads((test_dir / "metrics.json").read_text(encoding="utf-8"))
    if saved_metrics != metrics:
        raise RuntimeError("Lỗi: Đọc lại metrics không khớp.")
    print(json.dumps(results, ensure_ascii=False, indent=2))
    print("Thư mục kết quả:", test_dir)
    del model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Test chưa chạy; chưa tạo điểm đánh giá giả.")


## 8. Tải kết quả test

In [ ]:
if evaluate_test:
    from google.colab import files
    metrics_path = test_dir / "metrics.json"
    saved_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    if saved_metrics["run_id"] != evaluation_run_id or saved_metrics["run_status"] != "completed":
        raise RuntimeError("Lỗi: Chưa có kết quả test hoàn tất của run đã chọn.")
    archive_path = shutil.make_archive(str(test_dir), "zip", root_dir=test_dir)
    print("File tải về:", archive_path)
    files.download(archive_path)
else:
    print("Chưa bật test, không có ZIP test mới để tải.")
